# 04 sklearn 分类 Baseline：Iris 鸢尾花多分类

目标：用 sklearn 完成一个完整分类流程，从数据读取、可视化、划分数据、训练 baseline，到比较多个模型和做交叉验证。

这次用 Iris 鸢尾花数据集。它是机器学习入门中最经典的小数据集之一。

任务：根据花的 4 个测量值，预测鸢尾花属于 3 个类别中的哪一类。

## 1. 实验背景

Iris 数据集包含 150 朵鸢尾花，每朵花有 4 个特征：

| 特征 | 含义 |
|---|---|
| sepal length | 萼片长度 |
| sepal width | 萼片宽度 |
| petal length | 花瓣长度 |
| petal width | 花瓣宽度 |

目标类别有 3 个：

- setosa
- versicolor
- virginica

这是一个多分类任务，不是二分类。

## 2. 导入依赖

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

## 3. 读取 Iris 数据集

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature names:", feature_names)
print("Target names:", target_names)

## 4. 转成 DataFrame，做简单 EDA

EDA 是 Exploratory Data Analysis，意思是探索性数据分析。先看数据长什么样，再决定怎么建模。

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y
df["species"] = df["target"].map({i: name for i, name in enumerate(target_names)})

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["species"].value_counts().plot(kind="bar")
plt.xlabel("species")
plt.ylabel("count")
plt.title("Class Distribution")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.show()

## 5. 简单可视化

先用两个花瓣特征画散点图。Iris 数据集中，花瓣长度和花瓣宽度通常对分类很有用。

In [ ]:
x_feature = "petal length (cm)"
y_feature = "petal width (cm)"

for species_name in target_names:
    subset = df[df["species"] == species_name]
    plt.scatter(subset[x_feature], subset[y_feature], label=species_name, alpha=0.8)

plt.xlabel(x_feature)
plt.ylabel(y_feature)
plt.title("Iris: petal length vs petal width")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. 划分训练集和测试集

请你填写 `TEST_SIZE`。

提示：可以先用 `0.2`。因为数据很小，使用 `stratify=y` 保持每个类别比例一致。

In [ ]:
TEST_SIZE = None  # TODO: 你来填，例如 0.2

assert TEST_SIZE is not None, "请先填写 TEST_SIZE，例如 0.2"

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("train class counts:", np.bincount(y_train))
print("test class counts:", np.bincount(y_test))

## 7. 建立第一个 baseline：Logistic Regression

这里用 pipeline 把标准化和模型放在一起。

逻辑回归对特征尺度敏感，所以先 `StandardScaler()`。

In [ ]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_pred)

print(f"Logistic Regression test accuracy: {logistic_accuracy:.4f}")

In [ ]:
print(classification_report(y_test, logistic_pred, target_names=target_names))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    logistic_pred,
    display_labels=target_names,
    cmap="Blues",
)
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 8. 对比树模型

现在对比两个常见模型：

- Decision Tree：单棵决策树，容易解释，但可能过拟合。
- Random Forest：很多决策树投票，通常比单棵树更稳。

请你填写 `TREE_MAX_DEPTH`。

提示：可以先试 `2`、`3`、`None`。`None` 表示不限制树深度。

In [ ]:
TREE_MAX_DEPTH = None  # TODO: 你可以先填 3，也可以保留 None 观察是否过拟合

tree_model = DecisionTreeClassifier(
    max_depth=TREE_MAX_DEPTH,
    random_state=42,
)

forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=TREE_MAX_DEPTH,
    random_state=42,
)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: test accuracy = {acc:.4f}")

## 9. 交叉验证

Iris 数据集很小，单次 train/test 切分可能有偶然性。交叉验证会多次切分数据，得到更稳定的评估。

请你填写 `CV_FOLDS`。

提示：可以先用 `5`。

In [ ]:
CV_FOLDS = None  # TODO: 你来填，例如 5

assert CV_FOLDS is not None, "请先填写 CV_FOLDS，例如 5"

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=CV_FOLDS, scoring="accuracy")
    print(f"{name}")
    print(f"  scores: {scores}")
    print(f"  mean accuracy: {scores.mean():.4f}")
    print(f"  std: {scores.std():.4f}")
    print()

## 10. 特征重要性

树模型可以给出 feature importance，帮助我们初步判断哪些特征更重要。

In [ ]:
forest_model.fit(X_train, y_train)
importances = forest_model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("importance")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.25)
plt.show()

## 11. 总结

请你完成下面的问题：

1. Iris 是二分类还是多分类？为什么？
2. 哪两个特征在散点图中最容易把类别分开？
3. Logistic Regression、Decision Tree、Random Forest 哪个效果最好？结果是否可能受数据划分影响？
4. 为什么小数据集要看交叉验证，而不是只看一次 test accuracy？
5. 这次的 baseline 流程和前面手写逻辑回归有什么区别？

我的总结：

- TODO
- TODO
- TODO